# RQ3-RQ7 Ablation Matrix

This notebook implements the frozen grouped-CV exploration for RQ3-RQ7 using the split artifacts in `data/splits/`.
It delegates the heavy lifting to `src/python/rq3_rq7_ablation.py` so the workflow stays reproducible and compact.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / 'src').exists())
SRC_DIR = ROOT / 'src' / 'python'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rq3_rq7_ablation import run_analysis

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [2]:
results = run_analysis()
display(Markdown(f"Artifacts written to `{results['artifact_dir'].relative_to(ROOT)}`."))

Artifacts written to `artifacts/rq3_rq7_ablation`.

## Event Support

In [3]:
results['support_summary']

,family,cumul_support_share,last15_support_share
0,pass,0.983746,0.968551
1,pressure,0.924382,0.821555


## RQ3-RQ7 Mean CV Results

In [4]:
results['results']

,rq,variant,feature_blocks,feature_count,balanced_accuracy,roc_auc,pr_auc,brier_score
0,RQ3,direct_shots_sprints_plus_context,direct_shots_sprints + context,20,0.587742,0.624079,0.116955,0.219722
1,RQ3,full_base_plus_context,base_behavior + context,26,0.576889,0.613335,0.110121,0.219631
2,RQ4,base_plus_context,base_behavior + context,26,0.576889,0.613335,0.110121,0.219631
3,RQ4,base_plus_context_plus_pass,base_behavior + context + pass_event,34,0.558711,0.597795,0.108595,0.220299
4,RQ4,base_plus_context_plus_pressure,base_behavior + context + pressure_event,36,0.554010,0.592940,0.107521,0.216203
5,RQ4,base_plus_context_plus_pass_pressure,base_behavior + context + pass_event + pressur...,44,0.550415,0.583718,0.100753,0.216565
6,RQ5,last15_only_plus_context,last15 + context,17,0.602725,0.657976,0.124798,0.218872
7,RQ5,cumul_only_plus_context,cumul + context,17,0.587112,0.617521,0.115236,0.220096
8,RQ5,history_counts_only_plus_context,history_counts + context,15,0.581973,0.623831,0.128259,0.218090
9,RQ5,last15_plus_cumul_plus_context,last15 + cumul + context,26,0.576889,0.613335,0.110121,0.219631


## RQ3

In [5]:
display(results['results'][results['results']['rq'] == 'RQ3'])
display(Markdown(results['conclusions']['RQ3']))

,rq,variant,feature_blocks,feature_count,balanced_accuracy,roc_auc,pr_auc,brier_score
0,RQ3,direct_shots_sprints_plus_context,direct_shots_sprints + context,20,0.587742,0.624079,0.116955,0.219722
1,RQ3,full_base_plus_context,base_behavior + context,26,0.576889,0.613335,0.110121,0.219631


Direct sprint-and-shot features outperform the broader base model by +0.0109 Balanced Accuracy, so the leaner sprint-shot view is adequate in this CV setup.

## RQ4

In [6]:
display(results['rq4_delta'])
display(Markdown(results['conclusions']['RQ4']))

,rq,variant,feature_blocks,feature_count,balanced_accuracy,roc_auc,pr_auc,brier_score,balanced_accuracy_delta,roc_auc_delta
2,RQ4,base_plus_context,base_behavior + context,26,0.576889,0.613335,0.110121,0.219631,0.000000,0.000000
3,RQ4,base_plus_context_plus_pass,base_behavior + context + pass_event,34,0.558711,0.597795,0.108595,0.220299,-0.018177,-0.015540
4,RQ4,base_plus_context_plus_pressure,base_behavior + context + pressure_event,36,0.554010,0.592940,0.107521,0.216203,-0.022879,-0.020395
5,RQ4,base_plus_context_plus_pass_pressure,base_behavior + context + pass_event + pressur...,44,0.550415,0.583718,0.100753,0.216565,-0.026474,-0.029617


base_plus_context_plus_pass is the least harmful event extension, but it still shifts Balanced Accuracy by -0.0182 versus the base+context reference.

## RQ5

In [7]:
display(results['rq5_table'])
display(Markdown(results['conclusions']['RQ5']))

,rq,variant,feature_blocks,feature_count,balanced_accuracy,roc_auc,pr_auc,brier_score
0,RQ5,last15_only_plus_context,last15 + context,17,0.602725,0.657976,0.124798,0.218872
1,RQ5,cumul_only_plus_context,cumul + context,17,0.587112,0.617521,0.115236,0.220096
2,RQ5,history_counts_only_plus_context,history_counts + context,15,0.581973,0.623831,0.128259,0.218090
3,RQ5,last15_plus_cumul_plus_context,last15 + cumul + context,26,0.576889,0.613335,0.110121,0.219631


last15_only_plus_context is the strongest temporal specification under the frozen CV protocol.

## RQ6

In [8]:
display(results['rq6_coefficients'].head(20))
display(Markdown(results['conclusions']['RQ6']))

,feature,mean_coefficient,abs_mean_coefficient
0,ratio_mean_max_speed,1.225217,1.225217
1,ratio_peak_speed,0.986970,0.986970
2,excess_share_sprints,-0.910490,0.910490
3,excess_share_hsr,0.723557,0.723557
4,delta_vs_cumul_shots,-0.593235,0.613282
5,delta_vs_cumul_peak_speed,-0.560807,0.560807
6,ratio_hsr,-0.549720,0.549720
7,excess_share_shots_top_third,0.525734,0.525734
8,delta_vs_cumul_sprints,0.503934,0.503934
9,excess_share_shots_under_press,-0.477004,0.477004


Adding relative-intensity features reduces Balanced Accuracy by -0.0036; the strongest coefficient by magnitude is ratio_mean_max_speed (+1.2252).

## RQ7

In [9]:
display(results['rq7_importance'])
display(Markdown(results['conclusions']['RQ7']))

,feature,mean_importance_drop
0,position,0.118279
1,minutes_available_before_checkpoint,0.052144
2,formation,0.007622
3,minute_out,0.005162
4,subbed,0.003641
5,checkpoint,0.000462
6,is_home,-0.002559
7,minute_in,-0.004083


The context-only model leads the context-augmented behavior model by +0.0481 Balanced Accuracy, and position is the strongest context variable by mean ROC AUC permutation drop (0.1183).